In [ ]:
import random
import copy
import numpy as np
from catalystGA import GA, Ligand, Metal
from catalystGA.reproduction_utils import graph_crossover, graph_mutate
from catalystGA.utils import MoleculeOptions

## Define the Catalyst

Inherit from `BaseCatalyst` and implement the `calculate_score` method. 
`calculate_score` requires the first two arguments to be `n_cores` and `envvar_scratch`. Any additional arguments can be added as needed.
Needs to set attribute `self.score` to a float value representing the score of the catalyst.

In [ ]:
import logging
import math
from typing import List

from rdkit.Chem.Crippen import MolLogP
from catalystGA import BaseCatalyst
from rdkit import Chem
from rdkit.Chem import rdEHTTools, rdDistGeom


_logger = logging.getLogger("ExampleCatalyst")


METALS = "Pd,Fe,Cu,Ni,Ag,Au,Pt"
HEADER1 = ["Conf-ID", "GFN-2 OPT [Hartree]"]
ROW_FORMAT1 = "{:>15}{:>25}"


class ExampleCatalyst(BaseCatalyst):
    n_ligands = 2

    def __init__(self, metal: Chem.Mol, ligands: List):
        self.metal_partial_charge = math.nan
        super().__init__(metal, ligands)

    def calculate_score(
        self,
        n_cores,
        envvar_scratch,
        example_variable1=5.0,
        example_variable2=1.0,
    ):
        # envvar_scrartch is the name of the environment variable pointing to a scratch directory
        _logger.info(
            f"Calculating score for {self} with example_variable1={example_variable1} and example_variable2={example_variable2}"
        )
        # make 3d conformer
        mol3d = Chem.AddHs(self.mol)
        rdDistGeom.EmbedMolecule(mol3d, rdDistGeom.KDG())
        # here could be some xtb optimization, etc. but I skip that
        # calculate partial charge of metal atom
        passed, res = rdEHTTools.RunMol(mol3d)
        metal_partial_charge = res.GetAtomicCharges()[0]
        # add example_varibale 1 and deivide by example variable 2
        self.score = (-metal_partial_charge - example_variable2) / example_variable1
        _logger.info(
            f"Score for {self} is {self.score} with metal_partial_charge={metal_partial_charge}"
        )
        # setting attributes to be saved to database
        self.metal_partial_charge = metal_partial_charge


_logger.addHandler(logging.StreamHandler())
_logger.setLevel(logging.INFO)

In [ ]:
cat1 = ExampleCatalyst(
    metal=Metal("Pd"),
    ligands=[Ligand.from_smiles("c1ccccc1N"), Ligand.from_smiles("c1ccccc1O")],
)

In [ ]:
cat1.mol

## Create some random starting Catalysts

In [ ]:
metals_list = ["Pd"]
ligands_list = [
    "FP(F)F",
    "ClP(Cl)Cl",
    "CP(C)C",
    "C(C)(C)P(C(C)(C))C(C)(C)",
    "C(C)(C)(C)P(C(C)(C)(C))C(C)(C)(C)",
    "COP(CO)CO",
    "CCOP(OCC)OCC",
    "C1CCCC1P(C1CCCC1)C1CCCC1",
    "C1CCCCC1P(C1CCCCC1)C1CCCCC1",
    "c1ccccc1P(c1ccccc1)c1ccccc1",
    "c1cc(OC)ccc1P(c1ccc(OC)cc1)c1ccc(OC)cc1",
    "c1cc(Cl)ccc1P(c1ccc(Cl)cc1)c1ccc(Cl)cc1",
    "c1cc(F)ccc1P(c1ccc(F)cc1)c1ccc(F)cc1",
    "c1cc(C)ccc1P(c1ccc(C)cc1)c1ccc(C)cc1",
    "c1c(C)cccc1P(c1cc(C)ccc1)c1cc(C)ccc1",
    "c1(C)ccccc1P(c1c(C)cccc1)c1c(C)cccc1",
    "c1(C)cc(C)cc(C)c1P(c1c(C)cc(C)cc1(C))c1c(C)cc(C)cc1(C)",
    "C(C)(C)(C)P(C(C)(C)(C))C(C)(C)(C)",
    "C(C)(C)(C)P(CC(C)(C)(C))C(C)(C)(C)",
    "c1ccccc1P(C)C",
]

metals_list = [Metal(m) for m in metals_list]
ligands_list = [Ligand.from_smiles(s) for s in ligands_list]

In [ ]:
class GraphGA(GA):
    def __init__(
        self,
        mol_options: MoleculeOptions,
        population_size=5,
        n_generations=10,
        maximize_score=True,
        selection_pressure=1.5,
        mutation_rate=0.5,
        **kwargs,
    ):
        super().__init__(
            mol_options=mol_options,
            population_size=population_size,
            n_generations=n_generations,
            maximize_score=maximize_score,
            selection_pressure=selection_pressure,
            mutation_rate=mutation_rate,
            **kwargs,
        )

    def make_initial_population(self) -> List[ExampleCatalyst]:
        """Make initial population as a list of ExampleCatalysts."""
        population = []
        while len(population) < self.population_size:
            metal = random.choice(metals_list)
            ligands = random.choices(
                ligands_list, k=self.mol_options.individual_type.n_ligands
            )
            cat = self.mol_options.individual_type(metal, ligands)
            # remove duplicates
            if cat not in population:
                population.append(cat)
        return population

    @staticmethod
    def crossover(
        ind1: ExampleCatalyst, ind2: ExampleCatalyst
    ) -> ExampleCatalyst or None:
        """Crossover the graphs of two ligands of SuzukiCatalysts."""
        ind_type = type(ind1)
        # choose one ligand at random from ind1 and crossover with random ligand from ind2, then replace this ligand in ind1 with new ligand
        ind1_ligands = copy.deepcopy(ind1.ligands)
        new_mol = None
        counter = 0
        while not new_mol:
            idx1 = random.randint(0, len(ind1_ligands) - 1)
            idx2 = random.randint(0, len(ind2.ligands) - 1)
            new_mol = graph_crossover(ind1.ligands[idx1].mol, ind2.ligands[idx2].mol)
            counter += 1
            if counter > 10:
                return None
        try:
            Chem.SanitizeMol(new_mol)
            # this will catch if new_mol has no donor atom
            new_ligand = Ligand(new_mol)
            ind1_ligands[idx1] = new_ligand
            child = ind_type(ind1.metal, ind1_ligands)
            child.assemble()
            return child
        except Exception:
            return None

    @staticmethod
    def mutate(ind: ExampleCatalyst) -> ExampleCatalyst or None:
        """Mutate the graph of one ligand of a SuzukiCatalyst."""
        # pick one ligand at random, mutate and replace in ligand list
        idx = random.randint(0, len(ind.ligands) - 1)
        new_mol = None
        counter = 0
        while not new_mol:
            new_mol = graph_mutate(ind.ligands[idx].mol)
            counter += 1
            if counter > 10:
                return None
        try:
            Chem.SanitizeMol(new_mol)
            ind.ligands[idx] = Ligand(new_mol)
            ind.assemble()
            return ind
        except Exception:
            return None

In [ ]:
# Set Options for Molecule
mol_options = MoleculeOptions(
    individual_type=ExampleCatalyst,
    min_size=10,
    max_size=25,
    num_rotatable_bonds=8,
)

# Initialize GA
ga = GraphGA(
    mol_options=mol_options,
    population_size=8,
    n_generations=12,
)

In [ ]:
# Run the GA
results = ga.run()

## Analysis

In [ ]:
%%capture
! pip install matplotlib

In [ ]:
import matplotlib.pyplot as plt

generations = [r[0] for r in results]
mean_score = [np.mean([cat.score for cat in r[1]]) for r in results]

fig, ax = plt.subplots()
ax.plot(generations, mean_score, color="crimson", linewidth=2, marker="o")
ax.set_xlabel("Generation")
ax.set_ylabel("Mean Score")

In [ ]:
from rdkit.Chem import Draw

final_population = results[-1][1]

topN = 3
mols = [cat.mol for cat in final_population][:topN]
scores = [cat.score for cat in final_population][:topN]
metal_partial_charges = [cat.metal_partial_charge for cat in final_population][:topN]

Draw.MolsToGridImage(
    mols,
    legends=[
        f"Score: {scores[i]:.2f}\nMetal PC: {metal_partial_charges[i]:.2f}"
        for i in range(topN)
    ],
    subImgSize=(300, 400),
)